In [ ]:
import pandas as pd


portfolio = pd.read_csv('../../data/portfolio.csv')
profile = pd.read_csv('../../data/profile.csv')
transcript = pd.read_csv('../../data/transcript.csv')
menu = pd.read_csv('../../data/starbucks_menu_260112.csv')

In [4]:
import ast

# ----------------------------
# 0. 복사본
# ----------------------------
portfolio_df = portfolio.copy()
transcript_df = transcript.copy()

#  value → dict 변환
transcript_df['value_dict'] = transcript_df['value'].apply(ast.literal_eval)

#  value에서 값 추출
transcript_df['offer_id'] = transcript_df['value_dict'].apply(
    lambda x: x.get('offer id', x.get('offer_id'))
)
transcript_df['amount'] = transcript_df['value_dict'].apply(lambda x: x.get('amount'))
transcript_df['reward'] = transcript_df['value_dict'].apply(lambda x: x.get('reward'))


# customer_id 컬럼명 통일
if 'person' in transcript_df.columns:
    transcript_df = transcript_df.rename(columns={'person': 'customer_id'})

if 'id' in portfolio_df.columns:
    portfolio_df = portfolio_df.rename(columns={'id': 'offer_id'})

# ----------------------------
# 1. offer received 데이터 준비
# ----------------------------
offer_received = transcript_df[transcript_df['event'] == 'offer received'][
    ['customer_id', 'offer_id', 'time']
].copy()

offer_received = offer_received.rename(columns={'time': 'received_time'})

# 같은 고객이 같은 오퍼를 여러 번 받을 수 있으므로 순번 부여
offer_received['received_seq'] = (
    offer_received.groupby(['customer_id', 'offer_id']).cumcount() + 1
)

# duration 붙이기 (일 → 시간 변환용)
offer_received = offer_received.merge(
    portfolio_df[['offer_id', 'duration']],
    on='offer_id',
    how='left'
)

offer_received['offer_end_time'] = offer_received['received_time'] + offer_received['duration'] * 24

# ----------------------------
# 2. transaction 데이터 준비
# ----------------------------
transactions = transcript_df[transcript_df['event'] == 'transaction'][
    ['customer_id', 'time', 'amount']
].copy()

transactions = transactions.rename(columns={'time': 'transaction_time'})

# ----------------------------
# 3. 고객 기준으로 붙인 뒤,
#    오퍼 유효기간 내 transaction만 남기기
# ----------------------------
merged = offer_received.merge(
    transactions,
    on='customer_id',
    how='left'
)

valid_tx = merged[
    (merged['transaction_time'] >= merged['received_time']) &
    (merged['transaction_time'] <= merged['offer_end_time'])
].copy()

# ----------------------------
# 4. offer received별로 transaction 발생 여부 표시
# ----------------------------
# 하나의 offer received에 transaction이 1건 이상 있으면 conversion = 1
tx_flag = valid_tx.groupby(
    ['customer_id', 'offer_id', 'received_seq'],
    as_index=False
).agg(
    transaction_count=('transaction_time', 'count'),
    transaction_amount_sum=('amount', 'sum'),
    first_transaction_time=('transaction_time', 'min')
)

tx_flag['converted_to_transaction'] = 1

# offer_received 테이블에 결합
result = offer_received.merge(
    tx_flag[['customer_id', 'offer_id', 'received_seq', 'transaction_count',
             'transaction_amount_sum', 'first_transaction_time', 'converted_to_transaction']],
    on=['customer_id', 'offer_id', 'received_seq'],
    how='left'
)

# transaction 없는 경우 0 처리
result['converted_to_transaction'] = result['converted_to_transaction'].fillna(0).astype(int)
result['transaction_count'] = result['transaction_count'].fillna(0).astype(int)
result['transaction_amount_sum'] = result['transaction_amount_sum'].fillna(0)

# 최종 전환율 계산

total_received = len(result)
converted_received = result['converted_to_transaction'].sum()
conversion_rate = converted_received / total_received

print("전체 offer received 건수:", total_received)
print("transaction으로 이어진 건수:", converted_received)
print("offer received → transaction 전환율:", round(conversion_rate * 100, 2), "%")


전체 offer received 건수: 76277
transaction으로 이어진 건수: 59952
offer received → transaction 전환율: 78.6 %
